# MTUS1 transcript expression in cancer cell lines

## Overview

This notebook examines transcript-level expression of *MTUS1* isoforms in DepMap/CCLE cell-line models. The analysis focuses on the relative contribution of ATIP3-oriented transcripts in breast cancer and TNBC-relevant cell-line contexts, providing supplementary support for interpreting gene-level *MTUS1* analyses in an ATIP3-oriented breast cancer setting.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## 1. CCLE cell-line annotations

We use the file `Model_25Q3.csv`, which contains metadata for CCLE cell lines, 

You can access these files through DepMap's download page:  
👉 [https://depmap.org/portal/download/](https://depmap.org/portal/download/)


In [ ]:
# Load CCLE cell-line annotations.
meta_CCLE = pd.read_csv("../data/raw/Cell-lines/Model_25Q3.csv")

# Display available column names.
print(meta_CCLE.columns)
meta_CCLE.head()


In [ ]:
meta_CCLE[meta_CCLE['OncotreeCode'] == 'BRCA'] #22 cell lines   
meta_CCLE[meta_CCLE['OncotreeLineage'].str.contains('Breast', na=False)] #96 cell lines

## 2. RNA-seq transcript-level expression (CCLE, 25Q3)

We use the file `OmicsExpressionTranscriptTPMLogp1HumanAllGenesStranded.tsv`, which contains transcript-level RNA-seq expression data  
(log-transformed TPM values using RSEM) for CCLE cell lines.

- Each row represents a transcript (with `gene_id` and transcript ID as columns)
- Columns correspond to CCLE cell lines (`CCLE_ID`) and contain expression values (log2(TPM + 1))
- This dataset allows isoform-specific analysis of genes such as **MTUS1** (e.g., ATIP3, ATIP1, etc.)

🔗 **Original source**:  
[https://depmap.org/portal/download/](https://depmap.org/portal/download/) : *"Transcript expression (RSEM, transcript level)"*
**File to be downloaded**


In [ ]:
# Load CCLE transcript-level RNA-seq expression data. 
df_t = pd.read_csv("../data/raw/Cell-lines/OmicsExpressionTranscriptTPMLogp1HumanAllGenesStranded.csv")
df_t.head()


## 3. MTUS1/ATIP3


**Reference transcript (MTUS1 / ATIP3):**  
- Ensembl transcript: [ENST00000262102.10](https://www.ensembl.org/Homo_sapiens/Gene/Summary?db=core;g=ENSG00000129422;r=8:17643794-17801545;t=ENST00000693296)  
- UniProt entry: [Q9ULD2 (ATIP3)](https://www.uniprot.org/uniprotkb/Q9ULD2/entry#Q9ULD2-1)  
- This gene produces 7 isoforms via **alternative splicing** (ATIP1 to ATIP4, including ATIP3a and ATIP3b).

**Example isoforms of MTUS1 / ATIP variants in Human GRch38**

| Transcript ID  Human GRch38    | Name       | bp (length) | Protein (aa) | Biotype         | CCDS     | UniProt   | Synonym |    
|---------------------|------------|-------------|---------------|------------------|----------|-----------|---------|
| ENST00000693296.1  | MTUS1-230  | 6125         |  1270 aa       | Protein coding  | CCDS43717 | Q9ULD2-1  | ATIP3a  | 
| ENST00000262102.10  | MTUS1-201  | 6160        |  1270 aa       | Protein coding  | CCDS43717 | Q9ULD2-1  | ATIP3a  | 
| ENST00000381869.5   | MTUS1-204  | 6256        |  1216 aa       | Protein coding  | CCDS43716 | Q9ULD2-2  | ATIP3b  | 
| ENST00000519263.5   | MTUS1-216  | 4089        |  1216 aa       | Protein coding  | CCDS43716 | Q9ULD2-2  | ATIP3b  | 
| ENST00000297488.10  | MTUS1-202  | 3731        |  436 aa        | Protein coding  | CCDS43719 | Q9ULD2-3  | ATIP1   | 
|ENST00000634613.1.   |	MTUS1-229  | 1193     |	342aa	      |Protein coding   | CCDS83254 | Q9ULD2-4  |         |
| ENST00000381861.7   | MTUS1-203  | 3996        |  517 aa        | Protein coding  | CCDS43718 | Q9ULD2-6  | ATIP4   | 
|ENST00000544260.3	  | MTUS1-228  | 3650.       |	415aa	      | Protein coding. | CCDS55204 | Q9ULD2-7  |         |


> **Note:** ATIP2 (Q9ULD2-5) is expressed at very low levels in most tissues.


In [ ]:
# Filter the DataFrame to keep only entries for the MTUS1 gene.
# Remove rows containing only zeros.
df_t[['ModelID','ENST00000262102.10','ENST00000381869.5','ENST00000519263.5','ENST00000297488.10','ENST00000634613.1','ENST00000381861.7','ENST00000544260.3']] #ATIP3a 'ENST00000693296.1' not in index, ATIP3b, ATIP3b, ATIP1,Q9ULD2-4,ATIP4, Q9ULD2-7
# change column names for easier handling
df_t = df_t.rename(columns={'ENST00000262102.10': 'ATIP3a', 
                            'ENST00000381869.5': 'ATIP3b', 
                            'ENST00000519263.5': 'ATIP3b_2', 
                            'ENST00000297488.10': 'ATIP1', 
                            'ENST00000634613.1': 'Q9ULD2-4', 
                            'ENST00000381861.7': 'ATIP4', 
                            'ENST00000544260.3': 'Q9ULD2-7'})
df_t[['ModelID','ATIP3a','ATIP3b','ATIP3b_2','ATIP1','Q9ULD2-4','ATIP4','Q9ULD2-7']].head()

In [ ]:
# Merge transcript expression data with cell line metadata
df_merged = pd.merge(meta_CCLE, df_t, left_on='ModelID', right_on='ModelID')
df_merged.head()

In [ ]:
df_merged.value_counts('OncotreePrimaryDisease')[0:25]

In [ ]:
df_merged[df_merged.columns[0:10]]

In [ ]:
df_merged.value_counts('DepmapModelType')[20:30]

## 4. Expression of MTUS1 Transcripts by Cancer Type

To visualize the transcript-level expression of MTUS1 isoforms (e.g., ATIP1, ATIP3a, ATIP3b, ATIP4), we:

- Filter the RNA-seq dataset by TCGA tissue code (e.g., `OV`, `BRCA`, `LIHC`, etc.)
- Display a boxplot of transcript expression (log2(TPM + 1)) across all cell lines of the selected cancer type

This visualization helps compare the relative abundance of different MTUS1 isoforms within a specific cancer context.


In [ ]:
# Set the TCGA tissue code to filter cell lines (e.g., 'OV', 'BRCA','GBM','SKCM' etc.)
name_TCGA = 'UCEC'  # 'BRCA', 'LIHC', 'PRAD', 'STAD' can be used as alternatives 'BRCA' LUAD

# Filter RNA-seq data for the selected cancer type
df_2 = df_merged[df_merged['OncotreeCode'] == name_TCGA]
print("Number of cell lines:", df_2.shape[0])

# Plot the expression values using a boxplot
plt.figure(figsize=(9, 4))
# choose colors
colors = ['darkorchid', 'royalblue', 'magenta', 'lightgreen', 'brown', 'sandybrown', 'salmon']
sns.boxplot(data=df_2[['ATIP3a','ATIP3b','ATIP3b_2','ATIP1','Q9ULD2-4','ATIP4','Q9ULD2-7']], palette=colors)
plt.title(f'MTUS1 Transcript Expression in the {df_2.shape[0]} {name_TCGA} Cell Lines (CCLE)')
plt.ylabel('TPM (log1p transformed)')
plt.xlabel('Transcripts')
plt.xticks(rotation=45)



# ylim can be adjusted if needed
plt.ylim(0, 5)
plt.tight_layout()
plt.show()


In [ ]:
# Set the TCGA tissue code to filter cell lines (e.g., 'OV', 'BRCA','GBM','SKCM' etc.)
Disease = 'Uteri'  # 'Breast', 'Liver', 'Prostate', 'Stomach' can be used as alternatives 'Breast' Lung 'Glioma' Renal

df_2 = df_merged[df_merged['OncotreePrimaryDisease'].str.contains(Disease, na=False)]  # 'Breast', 'Liver', 'Prostate', 'Stomach' can be used as alternatives 'Breast' Lung
print("Number of cell lines:", df_2.shape[0])

# Plot the expression values using a boxplot
plt.figure(figsize=(9, 4))
colors = ['darkorchid', 'royalblue', 'magenta', 'lightgreen', 'brown', 'sandybrown', 'salmon']
sns.boxplot(data=df_2[['ATIP3a','ATIP3b','ATIP3b_2','ATIP1','Q9ULD2-4','ATIP4','Q9ULD2-7']], palette=colors)
plt.title(f'MTUS1 Transcript Expression in the {df_2.shape[0]} {Disease} Cell Lines (CCLE)')
plt.ylabel('TPM (log1p transformed)')
plt.xlabel('Transcripts')
plt.xticks(rotation=45)
# ylim can be adjusted if needed
#plt.ylim(0, 10)
plt.tight_layout()
plt.show()


In [ ]:
df_2.value_counts('DepmapModelType')

In [ ]:
df_2